In [ ]:
 #Libraray Import
import numpy as np
import pandas as pd
from sklearn.model_selection import (
    StratifiedKFold, LeaveOneOut, train_test_split, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.preprocessing import StandardScaler, RobustScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from collections import Counter
import time



In [ ]:
from google.colab import files

uploaded = files.upload()

Saving medical_data_clean.csv to medical_data_clean.csv


In [ ]:
# Assuming 'medical_data_clean.csv' was the uploaded file
df = pd.read_csv('medical_data_clean.csv')
display(df.head())

,S1_T1,S2_T1,S3_T1,S4_T1,S5_T1,S6_T1,S1_T2,S2_T2,S3_T2,S4_T2,...,S5_T3,S6_T3,Age,Sex,Smoking,LC_stage,LC_type,Group,Group_name,lscm
0,3.180000e-07,3.730000e-07,0.000230,9.420000e-07,5.440000e-07,7.010000e-07,6.790000e-07,6.920000e-07,0.000202,0.000004,...,8.850000e-07,7.660000e-07,50,1,1,3,0,1,LC group,4
1,3.300000e-07,2.770000e-07,0.000268,1.120000e-06,8.950000e-07,1.050000e-06,2.660000e-07,1.890000e-07,0.000095,0.000002,...,1.160000e-06,9.310000e-07,68,0,0,4,1,1,LC group,3
2,5.450000e-07,4.810000e-07,0.000527,2.400000e-06,2.260000e-06,2.310000e-06,7.360000e-07,5.290000e-07,0.000266,0.000005,...,4.080000e-06,3.070000e-06,70,1,1,1,1,1,LC group,4
3,3.780000e-07,4.190000e-07,0.000241,1.100000e-06,9.490000e-07,1.010000e-06,3.580000e-07,3.750000e-07,0.000095,0.000002,...,1.420000e-06,1.070000e-06,66,0,0,4,1,1,LC group,3
4,4.260000e-07,4.780000e-07,0.000276,1.440000e-06,8.630000e-07,1.050000e-06,4.350000e-07,4.680000e-07,0.000120,0.000003,...,1.400000e-06,1.190000e-06,69,1,1,1,1,1,LC group,4


In [ ]:
def nested_stratified_kfold_cv(X, y, n_outer_splits=5, n_inner_splits=3, random_state=42, use_smote=False):

    print("="*60)
    print(f"Nested CV: {n_outer_splits} outer folds x {n_inner_splits} inner folds")
    print("="*60)

    outer_skf = StratifiedKFold(n_splits=n_outer_splits, shuffle=True, random_state=random_state)
    inner_skf = StratifiedKFold(n_splits=n_inner_splits, shuffle=True, random_state=random_state)

    param_distributions = {
        'n_estimators': [50, 75, 100],
        'max_depth': [2,3, 4, 5],
        'learning_rate': [0.05, 0.1],
        'gamma': [0.0, 0.1],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'reg_lambda': [1, 1.5, 2]
    }

    outer_results = {
        'roc_auc': [],
        'accuracy': [],
        'sensitivity': [],   # Recall for positive class
        'specificity': [],
        'best_params': [],
        'inner_search_time': [], #Time for RandomizedSearchCV (inner loop)
        'fold_total_time': []   #Total time for each fold
    }

    X_values = X.values if isinstance(X, pd.DataFrame) else X
    y_values = y.values if isinstance(y, pd.Series) else y

    total_start_time = time.time()

    # ── OUTER LOOP ──────────────────────────────────────────────
    for outer_fold, (train_idx, test_idx) in enumerate(outer_skf.split(X_values, y_values), 1):

        outer_start_time = time.time() # Initialize outer_start_time at the beginning of each outer fold

        print(f"\n{'─'*60}")
        print(f"OUTER Fold {outer_fold}/{n_outer_splits}")
        print(f"{'─'*60}")

        X_train_outer, X_test_outer = X_values[train_idx], X_values[test_idx]
        y_train_outer, y_test_outer = y_values[train_idx], y_values[test_idx]

        # Scale using outer train set only
        scaler = RobustScaler()
        X_train_outer_scaled = scaler.fit_transform(X_train_outer)
        X_test_outer_scaled  = scaler.transform(X_test_outer)

        # Optional SMOTE applied only on outer training data
        if use_smote:
            smote = SMOTE(random_state=random_state)
            X_train_outer_scaled, y_train_outer = smote.fit_resample(X_train_outer_scaled, y_train_outer)

        # ── INNER LOOP: RandomizedSearchCV for hyperparameter tuning ──
        base_model = XGBClassifier(random_state=random_state, eval_metric='logloss')

        random_search = RandomizedSearchCV(
            estimator=base_model,
            param_distributions=param_distributions,
            n_iter=50,
            cv=inner_skf,          # Inner stratified folds
            scoring='roc_auc',
            n_jobs=-1,
            random_state=random_state,
            verbose=0,
            refit=True             # Refit best model on full outer-train set
        )

        inner_start_time = time.time()


        random_search.fit(X_train_outer_scaled, y_train_outer)
        inner_end_time = time.time()

        inner_search_time = inner_end_time - inner_start_time


        best_model  = random_search.best_estimator_
        best_params = random_search.best_params_

        print(f"  Best inner CV ROC-AUC : {random_search.best_score_:.4f}")
        print(f"  Best params           : {best_params}")
        print(f"  Inner CV Time         : {inner_search_time:.2f} seconds")

        # ── Evaluate best model on OUTER test fold ────────────────
        y_pred       = best_model.predict(X_test_outer_scaled)
        y_pred_proba = best_model.predict_proba(X_test_outer_scaled)[:, 1]

        # Confusion matrix components for sensitivity & specificity
        cm = confusion_matrix(y_test_outer, y_pred, labels=[0,1])
        tn,fp, fn, tp = cm.ravel()

        sensitivity  = tp / (tp + fn)   # Recall / True Positive Rate
        specificity  = tn / (tn + fp)   # True Negative Rate
        accuracy     = accuracy_score(y_test_outer, y_pred)
        roc_auc      = roc_auc_score(y_test_outer, y_pred_proba)
        fold_total_time = time.time() - outer_start_time

        outer_results['roc_auc'].append(roc_auc)
        outer_results['accuracy'].append(accuracy)
        outer_results['sensitivity'].append(sensitivity)
        outer_results['specificity'].append(specificity)
        outer_results['best_params'].append(best_params)
        outer_results['inner_search_time'].append(inner_search_time)
        outer_results['fold_total_time'].append(fold_total_time)

        print(f"  Outer Accuracy    : {accuracy:.4f}")
        print(f"  Outer ROC-AUC     : {roc_auc:.4f}")
        print(f"  Outer Sensitivity : {sensitivity:.4f}")
        print(f"  Outer Specificity : {specificity:.4f}")
        print(f"  Fold Total Time   : {fold_total_time:.2f} seconds")

    # ── AGGREGATE OUTER RESULTS ──────────────────────────────────
    mean_roc_auc     = np.mean(outer_results['roc_auc'])
    mean_accuracy    = np.mean(outer_results['accuracy'])
    mean_sensitivity = np.mean(outer_results['sensitivity'])
    mean_specificity = np.mean(outer_results['specificity'])

    std_roc_auc     = np.std(outer_results['roc_auc'])
    std_accuracy    = np.std(outer_results['accuracy'])
    std_sensitivity = np.std(outer_results['sensitivity'])
    std_specificity = np.std(outer_results['specificity'])

    print("\n" + "="*60)
    print("FINAL NESTED CV RESULTS (Outer Loop Means)")
    print("="*60)
    print(f"Mean ROC-AUC     : {mean_roc_auc:.4f} \u00b1 {std_roc_auc:.4f}")
    print(f"Mean Accuracy    : {mean_accuracy:.4f} \u00b1 {std_accuracy:.4f}")
    print(f"Mean Sensitivity : {mean_sensitivity:.4f} \u00b1 {std_sensitivity:.4f}")
    print(f"Mean Specificity : {mean_specificity:.4f} \u00b1 {std_specificity:.4f}")
    print("="*60)
    print("\nBest Parameters per Outer Fold:")
    for i, params in enumerate(outer_results['best_params'], 1):
        print(f"  Fold {i}: {params}")
    print("="*60)

    return outer_results

In [ ]:
#with only S4 sensors
X = df[["S4_T1", "S4_T2","S4_T3"]]
y = df["Group"]
display(X.head())
display(y.head())

,S4_T1,S4_T2,S4_T3
0,9.420000e-07,0.000004,0.000004
1,1.120000e-06,0.000002,0.000004
2,2.400000e-06,0.000005,0.000013
3,1.100000e-06,0.000002,0.000005
4,1.440000e-06,0.000003,0.000006


,Group
0,1
1,1
2,1
3,1
4,1


In [ ]:
nested_stratified_kfold_cv(X=X, y=y, n_outer_splits=5, n_inner_splits=3, random_state=42, use_smote=True)


Nested CV: 5 outer folds x 3 inner folds

────────────────────────────────────────────────────────────
OUTER Fold 1/5
────────────────────────────────────────────────────────────
  Best inner CV ROC-AUC : 0.9148
  Best params           : {'subsample': 0.9, 'reg_lambda': 1.5, 'n_estimators': 75, 'max_depth': 4, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.9}
  Inner CV Time         : 6.21 seconds
  Outer Accuracy    : 0.9583
  Outer ROC-AUC     : 0.9790
  Outer Sensitivity : 1.0000
  Outer Specificity : 0.9091
  Fold Total Time   : 6.24 seconds

────────────────────────────────────────────────────────────
OUTER Fold 2/5
────────────────────────────────────────────────────────────
  Best inner CV ROC-AUC : 0.9352
  Best params           : {'subsample': 0.9, 'reg_lambda': 1.5, 'n_estimators': 50, 'max_depth': 2, 'learning_rate': 0.1, 'gamma': 0.0, 'colsample_bytree': 0.9}
  Inner CV Time         : 3.28 seconds
  Outer Accuracy    : 0.8333
  Outer ROC-AUC     : 0.9510
  Outer 

{'roc_auc': [np.float64(0.979020979020979),
  np.float64(0.951048951048951),
  np.float64(0.9300699300699301),
  np.float64(0.8461538461538461),
  np.float64(0.9153846153846155)],
 'accuracy': [0.9583333333333334,
  0.8333333333333334,
  0.9166666666666666,
  0.8260869565217391,
  0.8260869565217391],
 'sensitivity': [np.float64(1.0),
  np.float64(0.7692307692307693),
  np.float64(0.9230769230769231),
  np.float64(0.8461538461538461),
  np.float64(0.7692307692307693)],
 'specificity': [np.float64(0.9090909090909091),
  np.float64(0.9090909090909091),
  np.float64(0.9090909090909091),
  np.float64(0.8),
  np.float64(0.9)],
 'best_params': [{'subsample': 0.9,
   'reg_lambda': 1.5,
   'n_estimators': 75,
   'max_depth': 4,
   'learning_rate': 0.1,
   'gamma': 0.1,
   'colsample_bytree': 0.9},
  {'subsample': 0.9,
   'reg_lambda': 1.5,
   'n_estimators': 50,
   'max_depth': 2,
   'learning_rate': 0.1,
   'gamma': 0.0,
   'colsample_bytree': 0.9},
  {'subsample': 0.8,
   'reg_lambda': 2,
  

In [ ]:
#without the S4_sensors
X1= df.drop(columns=["Group","LC_stage","LC_type","lscm","Group_name","S4_T1","S4_T2","S4_T3"])
display(X1.head())


,S1_T1,S2_T1,S3_T1,S5_T1,S6_T1,S1_T2,S2_T2,S3_T2,S5_T2,S6_T2,S1_T3,S2_T3,S3_T3,S5_T3,S6_T3,Age,Sex,Smoking
0,3.180000e-07,3.730000e-07,0.000230,5.440000e-07,7.010000e-07,6.790000e-07,6.920000e-07,0.000202,0.000002,1.500000e-06,5.560000e-07,2.720000e-07,0.000035,8.850000e-07,7.660000e-07,50,1,1
1,3.300000e-07,2.770000e-07,0.000268,8.950000e-07,1.050000e-06,2.660000e-07,1.890000e-07,0.000095,0.000001,8.760000e-07,2.110000e-07,1.750000e-07,0.000035,1.160000e-06,9.310000e-07,68,0,0
2,5.450000e-07,4.810000e-07,0.000527,2.260000e-06,2.310000e-06,7.360000e-07,5.290000e-07,0.000266,0.000003,2.750000e-06,6.980000e-07,5.250000e-07,0.000119,4.080000e-06,3.070000e-06,70,1,1
3,3.780000e-07,4.190000e-07,0.000241,9.490000e-07,1.010000e-06,3.580000e-07,3.750000e-07,0.000095,0.000001,9.480000e-07,2.760000e-07,3.300000e-07,0.000038,1.420000e-06,1.070000e-06,66,0,0
4,4.260000e-07,4.780000e-07,0.000276,8.630000e-07,1.050000e-06,4.350000e-07,4.680000e-07,0.000120,0.000001,1.170000e-06,3.210000e-07,3.700000e-07,0.000044,1.400000e-06,1.190000e-06,69,1,1


In [ ]:
nested_stratified_kfold_cv(X=X1, y=y, n_outer_splits=5, n_inner_splits=3, random_state=42, use_smote=True)

Nested CV: 5 outer folds x 3 inner folds

────────────────────────────────────────────────────────────
OUTER Fold 1/5
────────────────────────────────────────────────────────────
  Best inner CV ROC-AUC : 0.8959
  Best params           : {'subsample': 0.7, 'reg_lambda': 2, 'n_estimators': 50, 'max_depth': 5, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.8}
  Inner CV Time         : 5.61 seconds
  Outer Accuracy    : 0.8333
  Outer ROC-AUC     : 0.9231
  Outer Sensitivity : 0.8462
  Outer Specificity : 0.8182
  Fold Total Time   : 5.71 seconds

────────────────────────────────────────────────────────────
OUTER Fold 2/5
────────────────────────────────────────────────────────────
  Best inner CV ROC-AUC : 0.8997
  Best params           : {'subsample': 0.7, 'reg_lambda': 1.5, 'n_estimators': 100, 'max_depth': 2, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.9}
  Inner CV Time         : 4.88 seconds
  Outer Accuracy    : 0.9167
  Outer ROC-AUC     : 0.9930
  Outer S

{'roc_auc': [np.float64(0.923076923076923),
  np.float64(0.993006993006993),
  np.float64(0.923076923076923),
  np.float64(0.8153846153846154),
  np.float64(0.8307692307692307)],
 'accuracy': [0.8333333333333334,
  0.9166666666666666,
  0.75,
  0.782608695652174,
  0.8260869565217391],
 'sensitivity': [np.float64(0.8461538461538461),
  np.float64(1.0),
  np.float64(0.9230769230769231),
  np.float64(0.7692307692307693),
  np.float64(0.7692307692307693)],
 'specificity': [np.float64(0.8181818181818182),
  np.float64(0.8181818181818182),
  np.float64(0.5454545454545454),
  np.float64(0.8),
  np.float64(0.9)],
 'best_params': [{'subsample': 0.7,
   'reg_lambda': 2,
   'n_estimators': 50,
   'max_depth': 5,
   'learning_rate': 0.1,
   'gamma': 0.1,
   'colsample_bytree': 0.8},
  {'subsample': 0.7,
   'reg_lambda': 1.5,
   'n_estimators': 100,
   'max_depth': 2,
   'learning_rate': 0.1,
   'gamma': 0.1,
   'colsample_bytree': 0.9},
  {'subsample': 0.7,
   'reg_lambda': 1.5,
   'n_estimators'